<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/02-tool-calling.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 工具调用

**目标：** 端到端跑通 function/tool calling，包括出错路径。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，粘贴其值，并打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整讲解与选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型——见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 为什么要用工具，以及那一次往返

模型只知道训练数据里的东西。问它东京今天的天气，或者 `(17.5% of 2340)`，它会*猜*——猜得理直气壮，还经常猜错。**工具调用**填上了这个缺口：你交给模型一组它可以请求你运行的函数（打天气 API、做精确算术、查你的数据库），于是实时数据和真实计算流进答案，而不是被幻觉出来。这就是每一个「agent」背后的机制：模型伸向世界的手，恰恰就是你给它的那些工具。

关键的心智模型：**工具调用是一个协议，不是魔法。模型从不执行任何东西。** 它发出一个调用工具的*请求*，剩下的事由你的代码完成：

1. 你随用户消息一起发送 `tools=[...]`。
2. 模型回复时 `finish_reason == 'tool_calls'`，并在 `message.tool_calls` 上带一个或多个 tool call（工具名 + JSON 参数字符串 + 一个 `id`）。
3. **你**执行那个函数，然后发回一条 `role: "tool"` 消息，结果里引用那个 `id`。
4. 模型继续，可能还会调用更多工具，直到它用纯文本作答（`finish_reason == 'stop'`）。

这个 API 是无状态的：每一步都会重发整段对话。本 notebook 用两个工具：一个假的 `get_weather`（预置数据，让 notebook 在任何地方都能跑）和一个真实的 `calculate`，安全地求算术表达式的值。


In [ ]:
import ast
import json
import operator

TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': (
                'Get the current weather for a city. Call this whenever the user asks about '
                'weather, temperature, or outdoor conditions in a specific place.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {'type': 'string', 'description': 'City name, e.g. "Tokyo"'},
                    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']},
                },
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': (
                'Evaluate an arithmetic expression exactly. Call this for any math beyond '
                'trivial mental arithmetic instead of computing in your head. '
                'Supports + - * / ** and parentheses on numbers only.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'expression': {'type': 'string', 'description': 'e.g. "(17.5 * 12) / 3"'},
                },
                'required': ['expression'],
            },
        },
    },
]


# --- 具体实现 ---

FAKE_WEATHER = {
    'tokyo': {'temp_c': 31, 'conditions': 'humid, partly cloudy'},
    'paris': {'temp_c': 24, 'conditions': 'clear'},
    'london': {'temp_c': 18, 'conditions': 'light rain'},
}


def get_weather(city, unit='celsius'):
    data = FAKE_WEATHER.get(city.lower())
    if data is None:
        raise KeyError(f'no weather data for {city!r}')
    temp = data['temp_c'] if unit == 'celsius' else round(data['temp_c'] * 9 / 5 + 32)
    return f"{city}: {temp} degrees {unit}, {data['conditions']}"


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculate(expression):
    """通过 AST 求算术表达式的值——绝不要 eval() 模型提供的字符串。"""
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f'unsupported expression: {expression!r}')
    return ev(ast.parse(expression, mode='eval'))


HANDLERS = {'get_weather': get_weather, 'calculate': calculate}


## 第 1 步：看看原始的 tool 请求

在动手搭循环之前，先看看模型到底发回了什么。在 OpenAI 兼容接口上，`message.tool_calls` 里的每一项都有 `tc.function.name`、`tc.function.arguments`（一个用 `json.loads` 解析的 JSON**字符串**，不是 dict），以及 `tc.id`——你回传结果时要带上的那个句柄。

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=500,
    tools=TOOLS,
    messages=[{'role': 'user', 'content': 'What is the weather in Tokyo right now?'}],
)

print('finish_reason:', response.choices[0].finish_reason)
msg = response.choices[0].message
if msg.content:
    print('text:', msg.content)
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f'tool_call: {tc.function.name}({tc.function.arguments})  id={tc.id}')


跑一下，注意 `finish_reason` 是 `tool_calls`：模型在回合中间停住了，等你。tool call 之前也可能有一小段文本；这很正常。

## 第 2 步：完整循环

现在把回路闭合。有三条规则大多数教程都讲对了，还有两条它们会略过：

- 在结果之前先追加 assistant 的**整条** message 对象；`tool_calls` 列表必须留在历史里。
- 把**所有** tool 结果作为各自独立的 `role: "tool"` 消息返回，每一条都用 `tool_call_id` 匹配。
- 给循环设上限。一个不停调用工具的模型应该撞上你的上限，而不是你的信用卡。

教程略过的：异常（把 handler 包起来；返回一个错误字符串，让模型能优雅恢复）和并行调用（一个响应里可能包含*多个* tool call，所以要处理全部，而不只是第一个）。


In [ ]:
def execute_tool(tool_call):
    """运行一个 tool call；返回一条 tool 消息（错误字符串或结果）。"""
    try:
        handler = HANDLERS[tool_call.function.name]
        args = json.loads(tool_call.function.arguments)
        result = handler(**args)
        content = str(result)
        is_error = False
    except Exception as e:
        content = f'{type(e).__name__}: {e}'
        is_error = True
    return {'role': 'tool', 'tool_call_id': tool_call.id, 'content': content}, is_error


def run_agent(user_message, max_turns=5, verbose=True):
    messages = [{'role': 'user', 'content': user_message}]
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=600, tools=TOOLS, messages=messages,
        )
        choice = response.choices[0]
        if choice.finish_reason != 'tool_calls':
            return choice.message.content or ''

        # 追加 assistant 消息（为了历史必须保持 tool_calls 完整）
        messages.append(choice.message)

        # 执行所有 tool call，每个都作为独立的 tool 消息追加
        for tc in choice.message.tool_calls:
            tool_msg, is_error = execute_tool(tc)
            if verbose:
                flag = ' [ERROR]' if is_error else ''
                print(f"  turn {turn}: {tc.function.name}({tc.function.arguments}) "
                      f"-> {tool_msg['content']}{flag}")
            messages.append(tool_msg)

    raise RuntimeError(f'agent did not finish within {max_turns} turns')


print(run_agent('What is the weather in Tokyo, and what is 17.5% of 2340?'))


## 出错路径 1：工具抛异常

问一个我们那个假天气源不认识的城市。`get_weather` 抛出 `KeyError`，`execute_tool` 接住它并返回一条携带错误文本的 `role: "tool"` 消息（带 `is_error=True`，好让 trace 能标记它），然后由模型决定怎么办：道歉、请求澄清，或者换个办法。跑一下，注意模型给出了优雅的回答，而不是让循环崩掉。

In [ ]:
print(run_agent('What is the weather in Reykjavik?'))


## 出错路径 2：参数非法

模型也可能把*你的*工具调错：表达式你的计算器不支持、字段缺失、类型不对。两层防御，而且它们互补而非二选一：

| 防御 | 能抓到 | 抓不到 | 支持情况 | 用来做什么 |
|---|---|---|---|---|
| **`'strict': True` schema**（API 侧） | 形状错误：必填字段缺失、类型不对、未知 key | 符合 schema 但*语义*上错的取值 | Groq 各模型参差不齐；当成额外奖励 | 对结构做廉价的第一道过滤 |
| **你的 handler 自行校验**（代码） | 任何东西：未定义变量、越界值、业务规则 | 抓不到，因为这就是你的代码 | 始终可用 | 真正的保证；无论如何都要保留 |

- **你的 handler 自行校验。** `calculate` 会拒绝任何不是纯算术的东西（运行下面的单元格：模型会转述工具的抱怨，或者用修正过的表达式重试）。
- **API 可以替你校验。** 有些厂商支持在工具定义上加 `'strict': True`（配合 `additionalProperties: False` 和一个 `required` 列表），这样参数在到达你的代码之前就保证符合 schema。Groq 各模型的支持参差不齐，所以把它当额外奖励，而不是保证。而且 schema 校验抓不到*形状对但值错*的情况（一个能解析却引用了未定义变量的表达式），所以无论如何 handler 的检查都要保留。

In [ ]:
# 'x' 未定义，所以 AST 求值器会拒绝这个表达式。
# 跑一下，看这个错误如何往返：is_error -> 模型恢复或解释。
print(run_agent('Use the calculate tool to evaluate "x + 2" where x is 5.'))


## 并行 tool call

默认情况下，模型可能在一个响应里请求好几个工具：一条 assistant 消息的 `tool_calls` 列表里装着多项。我们的循环已经处理了这一点。它遍历每一项，为每个调用追加一条 `role: "tool"` 消息，各自用 `tool_call_id` 匹配回去。这种完整性是承重的：assistant 消息里的每个 tool call 都必须在下一次请求之前拿到匹配的结果，否则 API 会以对话格式非法为由拒绝。

In [ ]:
# 一个会一次性引出三次 tool call 的问题。看看每回合实际发生了几次。
print(run_agent(
    'Compare the current weather in Tokyo, Paris, and London. '
    'Then tell me the average of their temperatures in celsius (use calculate).'
))


跑一下，注意 trace 里的回合编号：三次 `get_weather` 调用通常落在*同一个*回合（并行），然后 `calculate` 在下一个回合才跟上，因为它依赖前者的结果。模型自己理清了这个依赖顺序。

## 工具描述就是 prompt

工具定义里杠杆最高的一行就是 `description`。模型几乎完全依据它来决定*是否*以及*如何*调用你的工具。含糊的描述会导致错误的调用：

- `"Gets weather"` 触发不足（模型凭记忆作答），还容易招来错误的参数（用国家而不是城市）。
- `"Get the current weather for a city. Call this whenever the user asks about weather... "` 说明了*何时*调用它，而不只是它做什么。描述里的触发条件在当前的模型上能实实在在地提高调用率，因为这类模型对伸手用工具偏保守。

和写 API 文档一样的纪律：描述每个参数、对封闭集合用 `enum`、只把真正必填的字段标为 required。如果模型一直误用某个工具，先去修描述，再去修 prompt。


## 最佳实践与反模式

| ✅ 要做 | ❌ 反模式 |
|---|---|
| 用 `json.loads` 解析 `tc.function.arguments`；追加 assistant 消息*和*每个调用对应的一条 `role:"tool"` 结果 | 漏掉某个 tool 结果，或改动 assistant 消息，导致 API 拒绝下一次请求 |
| 把 handler 包起来；返回一个能让模型恢复的*有教学意义*的错误字符串 | 让 handler 异常把循环搞崩 |
| 处理一个响应里的**所有** tool call（并行调用很正常） | 只处理 `tool_calls[0]`，丢掉其余 |
| 用 `max_turns` 给循环设上限 | 拿付费 key 跑无上限的工具循环 |
| 写清*何时*调用的描述；在代码里校验参数 | 含糊的描述；只信 `strict:true`（支持参差不齐） |

（完整全栈清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. 加第三个工具 `convert_currency(amount, from_ccy, to_ccy)`，用一个硬编码的汇率表支撑，然后问一个同时需要天气 + 汇率 + 算术的问题。在 trace 里检查并行情况。
2. 让 `get_weather` 有 50% 的概率随机抛出 `TimeoutError`，并扩展 `execute_tool`，在返回 `is_error` 之前先重试一次。对比有重试和没有重试的完整记录。
3. 给两个工具定义都加上 `'strict': True`（再加上 `additionalProperties: False` 和完整的 `required` 列表），然后试着制造一次畸形调用。确认不合形状的参数不会再到达你的 handler。
4. 把 `calculate` 的描述改到最含糊（「does math stuff」），把那个百分比问题重跑 3 次。数一数模型有多少次在心里算，而没有调用工具。
